# 00 — Setup & Smoke Test
**Objective:** prove Spark 4.2 runs on the standalone cluster and the ObjectScale PoC environment is configured.

**Customer sees:** versions, cluster state, and a real Spark job.
**TC:** TC0.1

## Step 1 — Start the session
Loads `.env`, builds a SparkSession on `spark://192.168.1.40:7077`, prints masked credentials and versions.

In [ ]:
import pandas as pd
from IPython.display import HTML, display
from poc import config, results, viz, s3util
import poc.spark_helpers as sh
spark = sh.build_session("obs-poc")
viz.table(__import__("pandas").DataFrame(sh.versions(spark).items(), columns=["component","version"]))
print("users:", config.masked_users())
print("Spark UI:", config.SPARK_UI)

**Presenter notes** (hidden)

Point out: no secrets printed — all keys masked. Show Spark UI at :8080.

## Step 2 — Cluster smoke job
`spark.range(10M).count()` runs a real distributed job on the worker and returns the count — proves the cluster computes.

In [ ]:
import time, pandas as pd
t0 = time.time(); n = spark.range(10_000_000).count(); dt = time.time() - t0
res = results.write("TC0.1", "spark.range(10M).count()", n == 10_000_000,
                    {"count": n, "seconds": round(dt, 1)})
viz.table(pd.DataFrame([{"count": n, "seconds": round(dt, 1)}]))
viz.banner(res["passed"], "TC0.1 smoke", f"{n:,} rows in {dt:.1f}s")
spark.stop()

**Presenter notes** (hidden)

If this is slow, the worker may still be registering — wait a few seconds.